# Customer Segmentation Analysis

## Overview
This notebook demonstrates the application of clustering algorithms to segment customers based on purchasing behavior, demographics, and engagement metrics. Customer segmentation enables targeted marketing strategies and personalized service offerings.

## Dataset
We'll be using the [Mall Customer Segmentation Data](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python) from Kaggle, which contains basic data about mall customers including Customer ID, age, gender, annual income, and spending score.

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Set visualization style
sns.set(style='whitegrid')
plt.style.use('seaborn-whitegrid')
%matplotlib inline

In [2]:
# Load the dataset
df = pd.read_csv('Mall_Customers.csv')

# Display the first few rows
df.head()

## Data Exploration and Preprocessing

In [3]:
# Check basic information about the dataset
print("Dataset Shape:", df.shape)
print("\nDataset Information:")
df.info()
print("\nSummary Statistics:")
df.describe()

In [4]:
# Check for missing values
print("Missing Values:")
df.isnull().sum()

In [5]:
# Convert categorical variable 'Gender' to numerical
df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})

# Display the updated dataframe
df.head()

## Exploratory Data Analysis

In [6]:
# Distribution of Age
plt.figure(figsize=(10, 6))
sns.histplot(df['Age'], kde=True)
plt.title('Distribution of Customer Age', fontsize=15)
plt.xlabel('Age', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.show()

In [7]:
# Distribution of Annual Income
plt.figure(figsize=(10, 6))
sns.histplot(df['Annual Income (k$)'], kde=True)
plt.title('Distribution of Annual Income', fontsize=15)
plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.show()

In [8]:
# Distribution of Spending Score
plt.figure(figsize=(10, 6))
sns.histplot(df['Spending Score (1-100)'], kde=True)
plt.title('Distribution of Spending Score', fontsize=15)
plt.xlabel('Spending Score (1-100)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.show()

In [9]:
# Gender distribution
plt.figure(figsize=(8, 6))
gender_counts = df['Gender'].value_counts()
plt.pie(gender_counts, labels=['Female', 'Male'], autopct='%1.1f%%', startangle=90, colors=['#ff9999','#66b3ff'])
plt.title('Gender Distribution', fontsize=15)
plt.axis('equal')
plt.show()

In [10]:
# Relationship between Annual Income and Spending Score
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Annual Income (k$)', y='Spending Score (1-100)', hue='Gender', data=df, palette=['blue', 'red'])
plt.title('Annual Income vs Spending Score', fontsize=15)
plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score (1-100)', fontsize=12)
plt.legend(title='Gender', labels=['Male', 'Female'])
plt.show()

## Feature Selection and Scaling

In [11]:
# Select features for clustering
X = df[['Annual Income (k$)', 'Spending Score (1-100)']]

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Determining Optimal Number of Clusters using Elbow Method

In [12]:
# Elbow Method to find optimal number of clusters
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Plot the Elbow Method graph
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--')
plt.title('Elbow Method for Optimal Number of Clusters', fontsize=15)
plt.xlabel('Number of Clusters', fontsize=12)
plt.ylabel('WCSS (Within-Cluster Sum of Squares)', fontsize=12)
plt.grid(True)
plt.show()

Based on the elbow method, we can see that the optimal number of clusters appears to be 5, as the rate of decrease in WCSS slows down significantly after this point.

## Silhouette Analysis for Cluster Validation

In [13]:
# Silhouette analysis to validate the number of clusters
silhouette_scores = []
for i in range(2, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=42)
    cluster_labels = kmeans.fit_predict(X_scaled)
    silhouette_avg = silhouette_score(X_scaled, cluster_labels)
    silhouette_scores.append(silhouette_avg)
    print(f"For n_clusters = {i}, the silhouette score is {silhouette_avg:.3f}")

# Plot silhouette scores
plt.figure(figsize=(10, 6))
plt.plot(range(2, 11), silhouette_scores, marker='o', linestyle='--')
plt.title('Silhouette Score for Different Number of Clusters', fontsize=15)
plt.xlabel('Number of Clusters', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.grid(True)
plt.show()

The silhouette analysis confirms that 5 clusters is a good choice, as it has one of the highest silhouette scores.

## K-Means Clustering with 5 Clusters

In [14]:
# Apply K-Means clustering with 5 clusters
kmeans = KMeans(n_clusters=5, init='k-means++', max_iter=300, n_init=10, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Get cluster centers and transform back to original scale
centers = scaler.inverse_transform(kmeans.cluster_centers_)

# Plot the clusters
plt.figure(figsize=(12, 8))
sns.scatterplot(x='Annual Income (k$)', y='Spending Score (1-100)', hue='Cluster', data=df, palette='viridis', s=100)

# Plot the centroids
plt.scatter(centers[:, 0], centers[:, 1], c='red', s=300, alpha=0.6, marker='*', label='Centroids')
plt.title('Customer Segments based on Annual Income and Spending Score', fontsize=15)
plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score (1-100)', fontsize=12)
plt.legend(title='Cluster')
plt.grid(True)
plt.show()

## Cluster Analysis and Interpretation

In [15]:
# Analyze each cluster
cluster_analysis = df.groupby('Cluster').agg({
    'Age': 'mean',
    'Annual Income (k$)': 'mean',
    'Spending Score (1-100)': 'mean',
    'Gender': ['count', lambda x: (x == 1).mean() * 100]  # Count and percentage of females
}).round(2)

# Rename columns for clarity
cluster_analysis.columns = ['Avg Age', 'Avg Annual Income (k$)', 'Avg Spending Score', 'Count', '% Female']
cluster_analysis

## Cluster Descriptions

Based on our analysis, we can describe the five customer segments as follows:

1. **Cluster 0: Middle Income, Average Spenders**
   - Average annual income: ~$55k
   - Average spending score: ~50
   - These customers have moderate income and average spending habits.

2. **Cluster 1: High Income, Low Spenders**
   - Average annual income: ~$88k
   - Average spending score: ~17
   - These customers have high income but are conservative in their spending.

3. **Cluster 2: Low Income, High Spenders**
   - Average annual income: ~$26k
   - Average spending score: ~82
   - These customers have lower income but spend more relative to their income.

4. **Cluster 3: High Income, High Spenders**
   - Average annual income: ~$86k
   - Average spending score: ~82
   - These are premium customers with high income and high spending habits.

5. **Cluster 4: Low Income, Low Spenders**
   - Average annual income: ~$27k
   - Average spending score: ~21
   - These customers have lower income and are conservative in their spending.

## Marketing Strategy Recommendations

Based on the customer segmentation, here are some targeted marketing strategies for each segment:

1. **Middle Income, Average Spenders (Cluster 0)**
   - Implement loyalty programs to increase spending frequency
   - Offer mid-range products with good value propositions
   - Use targeted promotions to encourage higher spending

2. **High Income, Low Spenders (Cluster 1)**
   - Focus on premium product quality and exclusivity
   - Emphasize value and investment aspects of products
   - Provide personalized shopping experiences to increase engagement

3. **Low Income, High Spenders (Cluster 2)**
   - Offer installment payment options
   - Create special discount programs for frequent shoppers
   - Develop budget-friendly product lines with trendy designs

4. **High Income, High Spenders (Cluster 3)**
   - Develop premium membership programs with exclusive benefits
   - Focus on luxury products and personalized services
   - Create VIP events and early access to new products

5. **Low Income, Low Spenders (Cluster 4)**
   - Implement aggressive discount strategies
   - Develop budget product lines with essential features
   - Use promotional offers to drive initial purchases

## Conclusion

This customer segmentation analysis has identified five distinct customer groups based on their annual income and spending patterns. Each segment represents a unique customer profile with different purchasing behaviors and preferences.

By understanding these segments, businesses can develop targeted marketing strategies, personalized product offerings, and tailored communication approaches to effectively engage with each customer group. This segmentation approach enables more efficient resource allocation and improved customer satisfaction through personalized experiences.

Future work could include:
- Incorporating additional customer attributes for more nuanced segmentation
- Implementing time-series analysis to track segment evolution
- Developing predictive models to forecast customer segment transitions
- A/B testing of targeted marketing campaigns for each segment